## Matrix Solver Reference Implementation

### Floating Point Test

In [ ]:
from ieee754 import double, IEEE754

# 直接获取双精度表示
print(double(13.375))

# 使用类获取更多信息
x = 13.375
a = IEEE754(x,precision=1) # 默认双精度
print(str(a)) # 二进制表示
print(a.hex()) # 十六进制表示
print(a.json()) # 详细信息(JSON)

0 10000000010 1010110000000000000000000000000000000000000000000000
0 10000010 10101100000000000000000
('41560000', ['0100', '0001', '0101', '0110', '0000', '0000', '0000', '0000'])
{'number': Decimal('13.375'), 'edge_case': False, 'sign_bit': '1', 'exponent_bits': 8, 'mantissa_bits': 23, 'total_bits': 32, 'sign': '0', 'scale': 3, 'scaled_number': 107, 'scaled_number_in_binary': '1101011', 'unable_to_scale': False, 'bias': 127, 'bias_in_binary': '01111111', 'exponent': '10000010', 'mantissa': '10101100000000000000000', 'mantissa_base_10': Decimal('0.67187500000000000000000'), 'result': '0 10000010 10101100000000000000000', 'hexadecimal': '41560000', 'hexadecimal_parts': ['0100', '0001', '0101', '0110', '0000', '0000', '0000', '0000'], 'converted_number': Decimal('13.375'), 'error': Decimal('0.000')}


In [3]:
a, b = 6.25,13.25
a_plus_b = a+b
aa = IEEE754(a, precision=1)
bb = IEEE754(b, precision=1)
rr = IEEE754(a_plus_b, precision=1)
print(f"A        hex : {aa.hex()[0]}")
print(f"B        hex : {bb.hex()[0]}")
print(f"Expected hex : {rr.hex()[0]}")

A        hex : 40C80000
B        hex : 41540000
Expected hex : 419C0000


In [55]:
a, b, c = 122.04, 0.06152, 0.237
r = a * b + c
aa = IEEE754(a, precision=1)
bb = IEEE754(b, precision=1)
cc = IEEE754(c, precision=1)
rr = IEEE754(r, precision=1)
print(f"A        hex : {aa.hex()[0]}")
print(f"B        hex : {bb.hex()[0]}")
print(f"C        hex : {cc.hex()[0]}")
print(f"Expected hex : {rr.hex()[0]}")
print(r)
print(a/b)

A        hex : 42F4147B
B        hex : 3D7BFC65
C        hex : 3E72B021
Expected hex : 40F7D63B
7.7449008
1983.7451235370613


### LU Decomposition

In [158]:
import numpy as np

# available floating-point primitives

cycles = 0

verbose = 3


# cycle: 0
def neg(v: float) -> float:
    return -v


# cycle: 16
def fma(a: float, b: float, c: float) -> float:
    global cycles
    global verbose
    cycles += 16
    if verbose > 2:
        print(f"[fma] {a:.3f} * {b:.3f} + {c:.3f} = {a * b + c:.3f}")
    return a * b + c


# cycle: 28
def div(a: float, b: float) -> float:
    global cycles
    global verbose
    cycles += 28
    if verbose > 2:
        print(f"[div] {a:.3f} / {b:.3f} = {a / b:.3f}")
    return a / b


def lu_decomp(A: np.ndarray):
    n = A.shape[0]
    # L = np.eye(n, dtype=float)
    # U = np.zeros((n, n), dtype=float)
    L = np.random.rand(n, n)
    U = np.random.rand(n, n)
    for j in range(n):
        for i in range(n):
            if i <= j:
                U[i, j] = neg(A[i, j])
                for k in range(i):
                    U[i, j] = fma(L[i, k], U[k, j], U[i, j])  # k < i
                U[i, j] = neg(U[i, j])
            else:  # i > j
                L[i, j] = neg(A[i, j])
                for k in range(j):
                    L[i, j] = fma(L[i, k], U[k, j], L[i, j])  # k < j; i > j => k < i
                L[i, j] = div(L[i, j], U[j, j])
                L[i, j] = neg(L[i, j])
    return L, U


def lu_decomp_more_hard(A: np.ndarray):
    n = A.shape[0]
    LU = np.random.rand(n, n)
    global cycles
    global verbose
    cycles = 0

    def fetch_A(i, j):
        global cycles
        global verbose
        cycles += 2
        if verbose > 2:
            print(f"fetch  A i={i} j={j} => {A[i, j]:.3f}")
        return A[i, j]

    def fetch_LU(i, j):
        global cycles
        global verbose
        cycles += 2
        if verbose > 2:
            print(f"fetch LU i={i} j={j} => {LU[i, j]:.3f}")
        return LU[i, j]

    def store_LU(i, j, v):
        global cycles
        global verbose
        cycles += 2
        if verbose > 2:
            print(f"store LU i={i} j={j} => {v:.3f}")
        LU[i, j] = v


    for idx_j in range(n):
        for idx_i in range(n):
            f32_1 = fetch_A(idx_i, idx_j)  # 2 cycles {StateFetchA + StateWaitA}
            f32_1 = -f32_1                              # combinational [f32_1_neg] {}
            idx_m = idx_j if idx_i > idx_j else idx_i   # combinational [min_i_j]
            for idx_k in range(idx_m):
                f32_2 = fetch_LU(idx_i, idx_k)  # 2 cycles {StateFetchL + StateWaitL}
                f32_3 = fetch_LU(idx_k, idx_j)  # 2 cycles {StateFetchU + StateWaitU}
                f32_1 = fma(f32_2, f32_3, f32_1)  # 16 cycles {StateStartFMA + StateWaitFMA}
            if idx_i > idx_j:
                f32_2 = fetch_LU(idx_j, idx_j)  # 2 cycles {StateFetchDiag + StateWaitDiag}
                f32_1 = div(f32_1, f32_2)  # 28 cycles {StateStartDiv + StateWaitDiv}
            f32_1 = -f32_1                              # combinational [f32_1_neg]
            store_LU(idx_i, idx_j, f32_1)  # 2 cycles {StateStore + StateWaitStore}

    print(f"cycle = {cycles}")

    return LU

StateIdle
StateFetchA
StateWaitA
StateInitK
StateFetchL
StateWaitL
StateFetchU
StateWaitU
StateStartFMA
StateWaitFMA
StateCheckK
StateFetchDiag
StateWaitDiag
StateStartDiv
StateWaitDiv
StateStore
StateWaitStore
StateAdvance
StateDone

In [ ]:
cycles = 0; mat = np.random.rand(64, 64); lu_decomp(mat); print(f"64x64 cycles={cycles}")
cycles = 0; mat = np.random.rand(32, 32); lu_decomp(mat); print(f"32x32 cycles={cycles}")
cycles = 0; mat = np.random.rand(16, 16); lu_decomp(mat); print(f"16x16 cycles={cycles}")
cycles = 0; mat = np.random.rand(8, 8); lu_decomp(mat); print(f"8x8 cycles={cycles}")

64x64 cycles=1421952
32x32 cycles=180544
16x16 cycles=23200
8x8 cycles=3024


In [ ]:
import scripts.matrix2mem

mat = np.random.rand(4, 4)
# mat = np.array(
#     [
#         [4.0, 3.0, 2.0, 1.0],
#         [3.0, 5.0, 1.5, -2.0],
#         [2.0, 1.5, 6.0, 0.5],
#         [1.0, -2.0, 0.5, 7.0],
#     ],
#     dtype=np.float32,
# )
print(f"mat:{mat}")
scripts.matrix2mem.matrix_to_word_mem(mat, "tmp/lu_test_input_4x4.mem")

mat:[[0.05386067 0.5107582  0.65246483 0.78647945 0.11108377 0.30971021
  0.79261606 0.15250532]
 [0.9902443  0.18920354 0.07490989 0.70992189 0.46129008 0.51192811
  0.65236981 0.59109966]
 [0.86797475 0.03297632 0.81262746 0.63891615 0.71016408 0.68355044
  0.15983112 0.42789689]
 [0.86245869 0.98812433 0.6483899  0.29100124 0.98891095 0.59101444
  0.49327079 0.91938402]
 [0.56444681 0.62272978 0.9331474  0.57870065 0.12422116 0.35616514
  0.33317975 0.95082343]
 [0.23242374 0.69942952 0.11728365 0.38740784 0.3952635  0.91243098
  0.19877884 0.13898275]
 [0.80821607 0.83968887 0.65995025 0.20018527 0.62243713 0.19168416
  0.80627367 0.24742911]
 [0.33457636 0.58968978 0.6334128  0.32884557 0.97608467 0.27441023
  0.88463996 0.58413467]]
Wrote 64 words to tmp\lu_test_input_4x4.mem
Matrix shape: (8, 8)
Flatten order: row-major


In [163]:
lu = lu_decomp_more_hard(mat)
uu = np.triu(lu)
ll = np.tril(lu, k=-1) + np.eye(lu.shape[0])
print(f"L:{ll}\nU:{uu}\nLU:{lu}")
print(f"L@U:{ll @ uu}")
scripts.matrix2mem.matrix_to_word_mem(lu, "tmp/lu_test_expected_4x4.mem")

fetch  A i=0 j=0 => 0.054
store LU i=0 j=0 => 0.054
fetch  A i=1 j=0 => 0.990
fetch LU i=0 j=0 => 0.054
[div] -0.990 / 0.054 = -18.385
store LU i=1 j=0 => 18.385
fetch  A i=2 j=0 => 0.868
fetch LU i=0 j=0 => 0.054
[div] -0.868 / 0.054 = -16.115
store LU i=2 j=0 => 16.115
fetch  A i=3 j=0 => 0.862
fetch LU i=0 j=0 => 0.054
[div] -0.862 / 0.054 = -16.013
store LU i=3 j=0 => 16.013
fetch  A i=4 j=0 => 0.564
fetch LU i=0 j=0 => 0.054
[div] -0.564 / 0.054 = -10.480
store LU i=4 j=0 => 10.480
fetch  A i=5 j=0 => 0.232
fetch LU i=0 j=0 => 0.054
[div] -0.232 / 0.054 = -4.315
store LU i=5 j=0 => 4.315
fetch  A i=6 j=0 => 0.808
fetch LU i=0 j=0 => 0.054
[div] -0.808 / 0.054 = -15.006
store LU i=6 j=0 => 15.006
fetch  A i=7 j=0 => 0.335
fetch LU i=0 j=0 => 0.054
[div] -0.335 / 0.054 = -6.212
store LU i=7 j=0 => 6.212
fetch  A i=0 j=1 => 0.511
store LU i=0 j=1 => 0.511
fetch  A i=1 j=1 => 0.189
fetch LU i=1 j=0 => 18.385
fetch LU i=0 j=1 => 0.511
[fma] 18.385 * 0.511 + -0.189 = 9.201
store LU i=1 